In [13]:
import sys
import pyomo
import pandas as pd

from pyomo.environ import *
solver = SolverFactory('cbc')


UC

In [14]:
from pyomo.environ import *

# =====================
# Model
# =====================
model = ConcreteModel()

# =====================
# Sets
# =====================
model.G = Set(initialize=['G1','G2','G3'])
model.T = Set(initialize=[1,2,3,4,5,6,7,8])

# =====================
# Parameters
# =====================
Pmin = {'G1':20, 'G2':30, 'G3':50}
Pmax = {'G1':100, 'G2':120, 'G3':150}
Cost = {'G1':15, 'G2':15, 'G3':12.35}

Demand = {1:150, 2:250, 3:220, 4:200.3, 5:300.4, 6:160, 7:340, 8:50}

model.Pmin = Param(model.G, initialize=Pmin)
model.Pmax = Param(model.G, initialize=Pmax)
model.Cost = Param(model.G, initialize=Cost)
model.Demand = Param(model.T, initialize=Demand)

# =====================
# Variables
# =====================
model.P = Var(model.G, model.T, domain=NonNegativeReals)
model.u = Var(model.G, model.T, domain=Binary)

# =====================
# Objective
# =====================
def obj_rule(m):
    return sum(m.Cost[g] * m.P[g,t] for g in m.G for t in m.T)

model.Obj = Objective(rule=obj_rule, sense=minimize)

# =====================
# Constraints
# =====================

# Power balance
def balance_rule(m, t):
    return sum(m.P[g,t] for g in m.G) == m.Demand[t]

model.Balance = Constraint(model.T, rule=balance_rule)

# Capacity upper bound
def max_rule(m, g, t):
    return m.P[g,t] <= m.Pmax[g] * m.u[g,t]

model.MaxCap = Constraint(model.G, model.T, rule=max_rule)

# Capacity lower bound
def min_rule(m, g, t):
    return m.P[g,t] >= m.Pmin[g] * m.u[g,t]

model.MinCap = Constraint(model.G, model.T, rule=min_rule)

# =====================
# Solve
# =====================
solver = SolverFactory('cbc')
results = solver.solve(model)

# =====================
# Format
# =====================
def fmt(x):
    if abs(x - round(x)) < 1e-6:
        return int(round(x))
    else:
        return round(x, 2)
    
# =====================
# Print Results
# =====================
print("Total System Cost =", fmt(value(model.Obj)))

rows = []

for t in model.T:
    g1_p = value(model.P['G1', t])
    g2_p = value(model.P['G2', t])
    g3_p = value(model.P['G3', t])
    
    g1_cost = g1_p * value(model.Cost['G1'])
    g2_cost = g2_p * value(model.Cost['G2'])
    g3_cost = g3_p * value(model.Cost['G3'])
    
    total_cost = g1_cost + g2_cost + g3_cost
    
    rows.append([
        t,
        fmt(value(model.Demand[t])),
        fmt(g1_p),
        fmt(g2_p),
        fmt(g3_p),
        fmt(g1_cost),
        fmt(g2_cost),
        fmt(g3_cost),
        fmt(total_cost)
    ])

df_hourly = pd.DataFrame(
    rows,
    columns=[
        "Hour",
        "Demand",
        "G1_P",
        "G2_P",
        "G3_P",
        "G1_Cost",
        "G2_Cost",
        "G3_Cost",
        "Total Cost"
    ]
)

print("\n=== Hourly Economic Dispatch Result ===")
print(df_hourly.to_string(index=False))

Total System Cost = 22172

=== Hourly Economic Dispatch Result ===
 Hour  Demand  G1_P  G2_P  G3_P  G1_Cost  G2_Cost  G3_Cost  Total Cost
    1   150.0   0.0   0.0   150      0.0        0   1852.5      1852.5
    2   250.0 100.0   0.0   150   1500.0        0   1852.5      3352.5
    3   220.0  70.0   0.0   150   1050.0        0   1852.5      2902.5
    4   200.3  50.3   0.0   150    754.5        0   1852.5      2607.0
    5   300.4 100.0  50.4   150   1500.0      756   1852.5      4108.5
    6   160.0  20.0   0.0   140    300.0        0   1729.0      2029.0
    7   340.0 100.0  90.0   150   1500.0     1350   1852.5      4702.5
    8    50.0   0.0   0.0    50      0.0        0    617.5       617.5
